# Classificação com Regressão Logística - Câncer de Mama (UCI Breast Cancer Wisconsin Diagnostic)
Classificação binária com saída de probabilidade, no mesmo espírito do app de crioterapia (predict + predict_proba).

Dataset: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
Baixe o arquivo `wdbc.data` e coloque na mesma pasta deste notebook (arquivo sem cabeçalho).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

# O arquivo original não tem cabeçalho, então montamos os nomes das colunas
partes = ["mean", "se", "worst"]
medidas = ["radius","texture","perimeter","area","smoothness","compactness",
           "concavity","concave_points","symmetry","fractal_dimension"]
colunas = ["id", "diagnosis"] + [f"{m}_{p}" for p in partes for m in medidas]

df = pd.read_csv("wdbc.data", header=None, names=colunas)
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})  # 1 = maligno, 0 = benigno
df.head()

In [ ]:
# Balanceamento das classes
sns.countplot(x="diagnosis", data=df, palette="Set2")
plt.title("Diagnóstico (0 = Benigno, 1 = Maligno)")
plt.show()

In [ ]:
# Correlação das variáveis 'mean' com o diagnóstico
cols_mean = [f"{m}_mean" for m in medidas]
plt.figure(figsize=(8,6))
sns.heatmap(df[cols_mean + ["diagnosis"]].corr(), annot=True, fmt=".2f", cmap="RdBu_r")
plt.title("Correlação (variáveis médias)")
plt.show()

In [ ]:
# Separa X e y, padroniza e divide treino/teste
X = df.drop(columns=["id", "diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Treina a regressão logística
modelo = LogisticRegression(max_iter=5000)
modelo.fit(X_train_scaled, y_train)

y_pred = modelo.predict(X_test_scaled)
print("Acurácia:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão")
plt.show()

In [ ]:
# Curva ROC
y_prob = modelo.predict_proba(X_test_scaled)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1], [0,1], "r--")
plt.xlabel("Falso Positivo")
plt.ylabel("Verdadeiro Positivo")
plt.title("Curva ROC")
plt.legend()
plt.show()

In [ ]:
# Classificação de um novo exame com probabilidade (mesmo padrão do bottle_app_crioterapia.py)
exemplo = X_test.iloc[[0]]
exemplo_scaled = scaler.transform(exemplo)

res = modelo.predict(exemplo_scaled)
prob = modelo.predict_proba(exemplo_scaled)

diagnostico = "Maligno" if res[0] == 1 else "Benigno"
print("Diagnóstico:", diagnostico)
print("Probabilidade [Benigno, Maligno]:", prob)